# 🕸️ Lab 06 · Springs, light switches, and agents that say "done"

**World Models course · Part A: Lecture 17 · Part B: Lectures 20–22 · HW3 application extension** &nbsp;|&nbsp; ⏱ about 55 min &nbsp;|&nbsp; 💻 CPU only

**Part A: Learning physics.** Balls connected by springs. You will teach a model *one rule for how neighbours pull each other*, then watch that rule work on a chain it has **never seen**. This is the core idea behind graph-network simulators such as DeepMind's GNS and the GraphCast weather model.

**Part B: Digital world state.** A room with a light you can't see unless you look. You will track a **belief** about hidden state, then catch an "agent" that claims *"Done ✅"* without checking. This is the central failure mode of today's tool-using AI agents.

🧩 challenges · 🔮 predictions · 🎛️ playgrounds. Blank or wrong answers never break the notebook.

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
import os
try:
    import ipywidgets as widgets
    _WIDGETS = not os.environ.get("GUIDE_NO_WIDGETS")
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
def _ref_pull(x):
    force = np.zeros_like(x)
    gap = x[1:] - x[:-1]
    force[:-1] += gap
    force[1:] -= gap
    return force
CHALLENGES["pull"] = dict(title="Neighbours pull on each other", reference=_ref_pull,
    cases=[np.array([0.0, 1.0, 0.5, -0.2]), np.array([2.0, 2.0])],
    hint="A link stretches by (position of the right ball − position of the left ball). Compare each ball with the one after it: <code>x[1:] - x[:-1]</code>.",
    why="Each spring only looks at its two ends. A <b>local, relational</b> rule like this is exactly what a graph neural network's message function learns.")

CHALLENGES["shared_rule"] = dict(title="Apply one learned rule to every ball",
    reference=lambda x, v, w_pull, w_vel: w_pull * _ref_pull(x) + w_vel * v,
    cases=[(np.array([0., 1., 0.5]), np.array([0.1, 0., -0.2]), 1.5, -0.08)],
    hint="Acceleration = w_pull × (neighbour pull on each ball) + w_vel × (each ball's velocity).",
    why="The <b>same two weights</b> are reused for every ball. This weight-sharing is why the rule transfers to longer chains.")

CHALLENGES["toggle"] = dict(title="Update belief after a toggle", reference=lambda belief: belief[::-1],
    cases=[np.array([0.2, 0.8]), np.array([0.5, 0.5])],
    hint="If P(off)=0.2 and P(on)=0.8 before a perfect toggle, what are they after? Swap them: <code>belief[::-1]</code>.",
    why="A belief is a probability for each hidden state. Actions move it around even when you observe nothing.")

CHALLENGES["look"] = dict(title="Update belief after looking", reference=lambda observation: np.eye(2)[observation],
    cases=[0, 1],
    hint="After seeing the light is ON (1), you're certain: P(off)=0, P(on)=1. <code>np.eye(2)[observation]</code> builds that row.",
    why="An observation collapses uncertainty. A filter (lab 05) does the soft version of this for noisy sensors.")

def _test_verify(fn):
    successes, claims = 0, 0
    for seed in range(300):
        room = LightRoom(seed, flaky=0.4)
        claim = fn(room)
        claims += claim
        successes += (claim and room.state == ON)
    false_claims = claims - successes
    ok = false_claims == 0 and claims > 250
    return ok, f"Across 300 flaky rooms your agent made {false_claims} false claims and {claims} claims in total. A verifying agent should make zero false claims."
def _ref_verify(room, max_tries=6):
    obs = room.step(LOOK)
    tries = 0
    while obs != ON and tries < max_tries:
        room.step(TOGGLE)
        obs = room.step(LOOK)
        tries += 1
    return obs == ON
CHALLENGES["verify"] = dict(title="Only claim what you observed", reference=_ref_verify, test=_test_verify,
    hint="Keep toggling and looking <b>while</b> the observation is not ON.",
    why="This is <b>ReAct</b>-style acting: act → observe the real result → decide. Coding agents run the tests; web agents re-read the page.")

QUIZZES["bigger"] = dict(predict=True, q="The rule was learned only from 5-ball chains. On a 9-ball chain it will…",
    options=["fail: it has never seen 9 balls", "work about as well: the rule is about neighbours, not about how many balls there are"],
    answer=1, explain="Because one rule is applied to every link, the model does not care how many links exist. <b>Relational structure</b> generalises to new sizes.")
QUIZZES["stiffer"] = dict(predict=True, q="Now the real springs become 60% stiffer, but the model is not retrained. Will it still predict well?",
    options=["Yes, it learned physics", "No, it learned the specific stiffness it was trained on"],
    answer=1, explain="The model learned the <i>structure</i> (neighbours pull) and a <i>number</i> (stiffness 1.5). New physics needs new data or a model that estimates stiffness from history. Structure generalises; constants don't.")
QUIZZES["claimed"] = dict(q="An agent's log says: “Toggled the light ✅”. The switch is known to fail sometimes. Is the light on?",
    options=["Yes, the log says so", "Unknown until someone LOOKs"],
    answer=1, explain="An action's <i>intended</i> effect is not its <i>actual</i> effect. Generated text such as “done” is not evidence. Only an observation of the environment is.")
print('✅ Setup complete. Scroll down and run the cells in order.')

---
# Part A · Learning physics that generalises 🧲

## 1 · A chain of balls on springs
Each ball has a position (how far it is displaced from rest) and a velocity. Neighbouring balls are joined by springs.

### 🧩 Challenge 1 · Neighbours pull on each other
For every spring between ball *i* and ball *i+1*, the stretch is `x[i+1] − x[i]`. The spring pulls ball *i* **toward** *i+1* (+stretch) and ball *i+1* back (−stretch).

In [ ]:
def neighbour_pull(x):
    force = np.zeros_like(x)
    gap = ___      # 🧩 stretch of every spring at once
    force[:-1] += gap            # each spring pulls its left ball to the right…
    force[1:]  -= gap            # …and its right ball to the left
    return force

neighbour_pull = check("pull", neighbour_pull)
print("positions:", [0.0, 1.0, 0.5], "→ pull:", neighbour_pull(np.array([0.0, 1.0, 0.5])))

<details><summary>🤔 <b>Need a hint?</b></summary>

Compare each ball with the next one, all at once with slicing.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>gap = x[1:] - x[:-1]      # 🧩 stretch of every spring at once</pre>

</details>

### The reference simulator
Acceleration = stiffness × pull − damping × velocity. We step it forward with small time steps (lab 01's Euler idea again).

In [ ]:
dt, STIFFNESS, DAMPING = 0.04, 1.5, 0.08

def simulate(n_balls=5, steps=150, stiffness=STIFFNESS, seed=0):
    r = np.random.default_rng(seed)
    x, v = r.normal(0, 0.3, n_balls), r.normal(0, 0.15, n_balls)     # a random shove to start
    xs, vs = [x.copy()], [v.copy()]
    for _ in range(steps):
        a = stiffness * neighbour_pull(x) - DAMPING * v
        v = v + dt * a
        x = x + dt * v
        xs.append(x.copy()); vs.append(v.copy())
    return np.array(xs), np.array(vs)

xs, vs = simulate(seed=1)
plt.figure(figsize=(8, 3))
for i in range(5):
    plt.plot(xs[:, i] + 1.2 * i, label=f"ball {i}")      # offset lines so they don't overlap
plt.xlabel("time step"); plt.yticks([]); plt.title("A 5-ball spring chain wobbling (each line = one ball)"); plt.show()

## 2 · Learn the rule from data 📚
We record 40 episodes and ask linear regression to explain each ball's acceleration using only **two inputs per ball**: its neighbour pull and its own velocity. Every ball contributes a training example to the **same** two weights.

For comparison, a **no-interaction** model may use only velocity, as if balls ignored each other.

In [ ]:
X, y = [], []
for seed in range(40):
    xs, vs = simulate(seed=seed)
    for t in range(len(xs) - 1):
        accel = (vs[t + 1] - vs[t]) / dt                       # what actually happened to each ball
        X.append(np.stack([neighbour_pull(xs[t]), vs[t]], axis=1))   # 2 inputs per ball
        y.append(accel)
X, y = np.concatenate(X), np.concatenate(y)

relational = LinearRegression(fit_intercept=False).fit(X, y)
no_interaction = LinearRegression(fit_intercept=False).fit(X[:, 1:], y)
w_pull, w_vel = relational.coef_
print(f"learned rule: accel = {w_pull:.3f} × pull {w_vel:+.3f} × velocity")
print(f"true rule:    accel = {STIFFNESS:.3f} × pull {-DAMPING:+.3f} × velocity")

### 🧩 Challenge 2 · Apply one learned rule to every ball

In [ ]:
def learned_accel(x, v, w_pull, w_vel):
    return ___     # 🧩 the same two weights for every ball

learned_accel = check("shared_rule", learned_accel)

def rollout(x0, v0, steps, accel_fn):
    x, v, out = x0.copy(), v0.copy(), [x0.copy()]
    for _ in range(steps):
        v = v + dt * accel_fn(x, v)
        x = x + dt * v
        out.append(x.copy())
    return np.array(out)

<details><summary>🤔 <b>Need a hint?</b></summary>

Multiply the pull by one weight and the velocity by the other, then add.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return w_pull * neighbour_pull(x) + w_vel * v     # 🧩 the same two weights for every ball</pre>

</details>

In [ ]:
quiz("bigger")

In [ ]:
quiz("stiffer")

In [ ]:
tests = [("same kind of chain", 5, STIFFNESS), ("bigger chain (9 balls)", 9, STIFFNESS), ("stiffer springs (2.4)", 5, 2.4)]
fig, axs = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)
for ax, (name, n, k) in zip(axs, tests):
    err_rel, err_none = [], []
    for seed in range(100, 120):                                      # 20 unseen test episodes
        truth, v_truth = simulate(n_balls=n, stiffness=k, seed=seed)
        rel = rollout(truth[0], v_truth[0], len(truth) - 1, lambda x, v: learned_accel(x, v, w_pull, w_vel))
        none = rollout(truth[0], v_truth[0], len(truth) - 1, lambda x, v: no_interaction.coef_[0] * v)
        err_rel.append(np.mean((rel - truth) ** 2, axis=1)); err_none.append(np.mean((none - truth) ** 2, axis=1))
    ax.plot(np.mean(err_rel, 0), label="learned relational rule"); ax.plot(np.mean(err_none, 0), label="no interaction")
    ax.set(title=name, xlabel="rollout step"); print(f"{name:>24}: final error relational {np.mean(err_rel, 0)[-1]:.4f} · no-interaction {np.mean(err_none, 0)[-1]:.4f}")
axs[0].set_ylabel("position error (MSE)"); axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()

**Reading the results honestly:**
* On 5 and 9 balls the error is essentially **zero**. That's expected: we handed the learner the exact right inputs (neighbour pull and velocity), so it could recover the true rule. Treat it as a sanity check, not a research result. Real graph simulators must *learn* the message function from raw positions, with noise.
* With stiffer springs the error jumps from ~0 to about 0.07. That's still far better than ignoring neighbours, but the simulator is no longer trustworthy.

### 🎛️ Playground · Stress-test the learned simulator
Change the real world (`real_stiffness`, `n_balls`) without retraining. Watch the middle ball: real (solid) vs model (dashed).

In [ ]:
def stress(real_stiffness=1.5, n_balls=5, seed=7):
    truth, v_truth = simulate(n_balls=n_balls, stiffness=real_stiffness, seed=seed)
    pred = rollout(truth[0], v_truth[0], len(truth) - 1, lambda x, v: learned_accel(x, v, w_pull, w_vel))
    mid = n_balls // 2
    plt.figure(figsize=(7, 2.8))
    plt.plot(truth[:, mid], lw=2, label="real middle ball"); plt.plot(pred[:, mid], "--", label="learned model")
    plt.title(f"trained on stiffness 1.5 and 5 balls · error {np.mean((pred - truth) ** 2):.4f}"); plt.legend(fontsize=8); plt.show()

playground(stress, real_stiffness=(0.5, 4.0, 0.1, 1.5), n_balls=(3, 30, 1, 5), seed=(0, 50, 1, 7))

---
# Part B · Hidden digital state and honest agents 💡

## 3 · The dark room
A light is **OFF (0)** or **ON (1)**. You can't see it unless you act:

| Action | Effect | You observe |
|---|---|---|
| `wait` | nothing | nothing |
| `toggle` | flips the light (a *flaky* switch sometimes fails) | nothing |
| `look` | nothing | the true state |

The start state is random, so before the first look a 50/50 belief is *correct*, not a failure.

In [ ]:
OFF, ON = 0, 1
WAIT, TOGGLE, LOOK = "wait", "toggle", "look"

class LightRoom:
    def __init__(self, seed, flaky=0.0):
        self.rng = np.random.default_rng(seed)
        self.state = int(self.rng.integers(2))        # hidden from the agent
        self.flaky = flaky                            # chance a toggle silently fails
    def step(self, action):
        if action == TOGGLE and self.rng.random() >= self.flaky:
            self.state = 1 - self.state
        return self.state if action == LOOK else None

### 🧩 Challenge 3 · Update belief after a (perfect) toggle
`belief = [P(off), P(on)]`

In [ ]:
def after_toggle(belief):
    return ___          # 🧩 a perfect toggle swaps the two probabilities

after_toggle = check("toggle", after_toggle)

<details><summary>🤔 <b>Need a hint?</b></summary>

Reverse the two-element array.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return belief[::-1]          # 🧩 a perfect toggle swaps the two probabilities</pre>

</details>

### 🧩 Challenge 4 · Update belief after looking

In [ ]:
def after_look(observation):
    return ___   # 🧩 certain: all probability on what we saw

after_look = check("look", after_look)

<details><summary>🤔 <b>Need a hint?</b></summary>

Seeing ON (1) should give [0, 1]; seeing OFF (0) should give [1, 0].

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return np.eye(2)[observation]   # 🧩 certain: all probability on what we saw</pre>

</details>

A switch that fails with probability `p` gives a mixed update: `(1 − p) × swapped + p × unchanged`. Let's compare a **belief tracker** (memory) with an agent that only trusts its *current* observation, on 300 random action sequences with a 20% flaky switch. We score with the **Brier score**: squared error of the probabilities (0 is perfect, lower is better), counted only after the first look.

In [ ]:
def track(room, actions, flaky):
    belief, seen_once, rows = np.array([0.5, 0.5]), False, []
    for a in actions:
        obs = room.step(a)
        if a == TOGGLE:
            belief = (1 - flaky) * after_toggle(belief) + flaky * belief
        if obs is not None:
            belief, seen_once = after_look(obs), True
        current_only = after_look(obs) if obs is not None else np.array([0.5, 0.5])
        if seen_once:
            truth = np.eye(2)[room.state]
            rows.append((np.sum((belief - truth) ** 2), np.sum((current_only - truth) ** 2)))
    return np.mean(rows, axis=0)

gen = np.random.default_rng(0)
scores = np.array([track(LightRoom(s, flaky=0.2), gen.choice([WAIT, TOGGLE, LOOK], 40), 0.2) for s in range(300)])
print(f"Brier score · belief tracker (memory): {scores[:, 0].mean():.3f}")
print(f"Brier score · current observation only: {scores[:, 1].mean():.3f}")

## 4 · The agent that says "done" ✅❓
**Task:** *make sure the light is ON, then report success.*

* **Naive agent:** looks once; if the light is off, toggles; reports success. It trusts that its action worked.
* **Verifying agent:** your job.

In [ ]:
quiz("claimed")

In [ ]:
def naive_agent(room):
    if room.step(LOOK) != ON:
        room.step(TOGGLE)          # "I toggled it, so it's on now"
    return True                    # always claims success

### 🧩 Challenge 5 · Only claim what you observed

In [ ]:
def verifying_agent(room, max_tries=6):
    obs = room.step(LOOK)
    tries = 0
    while ___ and tries < max_tries:   # 🧩 keep going until you have SEEN it on
        room.step(TOGGLE)
        obs = room.step(LOOK)                  # check the real result of the action
        tries += 1
    return obs == ON                           # claim success only if we observed it

verifying_agent = check("verify", verifying_agent)

<details><summary>🤔 <b>Need a hint?</b></summary>

Loop while the latest observation is not ON.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>while obs != ON and tries &lt; max_tries:   # 🧩 keep going until you have SEEN it on</pre>

</details>

### 🎛️ Playground · How flaky is the tool?

In [ ]:
def audit(flaky=0.3):
    rows = []
    for name, agent in [("naive", naive_agent), ("verifying", verifying_agent)]:
        claims = real = actions = 0
        for seed in range(1000):
            room = LightRoom(seed, flaky)
            original = room.step
            def counting_step(a, _orig=original):
                nonlocal actions
                actions += 1
                return _orig(a)
            room.step = counting_step
            claimed = agent(room)
            claims += claimed; real += claimed and room.state == ON
        rows.append((name, claims / 10, real / 10, (claims - real) / 10, actions / 1000))
    print(f"{'agent':>10} | claims success | actually on | FALSE claims | actions per task")
    for name, c, r, f, a in rows:
        print(f"{name:>10} | {c:13.1f}% | {r:10.1f}% | {f:11.1f}% | {a:6.2f}")
    plt.figure(figsize=(5, 2.6))
    plt.bar([r[0] for r in rows], [r[3] for r in rows], color=["tab:red", "tab:green"])
    plt.ylabel("% false 'done' claims"); plt.title(f"switch fails {flaky:.0%} of the time"); plt.show()

playground(audit, flaky=(0.0, 0.9, 0.05, 0.3))

---
## 5 · Recap and industry links 🏭

| You built | Research name | Where it's used |
|---|---|---|
| One neighbour rule for all balls | graph network simulator, message passing | DeepMind **GNS** and **GraphCast** weather; NVIDIA **PhysicsNeMo** surrogate simulators |
| Worked on bigger chains, failed on stiffer springs | structural vs. parametric generalisation | why robot world models need diverse physics data (NVIDIA Cosmos, domain randomisation) |
| Belief over hidden state | partially observable state tracking | LLMs tracking game or UI state (Othello-GPT probing, lecture 20) |
| Verify-before-claim loop | ReAct, tool feedback, execution checks | Claude Code / Codex running tests, web agents re-reading pages, SWE-bench harnesses |

### 🧪 HW3 application extension, beginner version
Pick **one**:
* **Physics:** add a little noise to the recorded accelerations. How many training episodes before `w_pull` is accurate to 2 decimal places?
* **Digital:** add a second hidden variable (e.g. a door that is locked or unlocked, and `toggle` only works when unlocked). Extend the belief and the verifying agent.

### 🗣️ Explain it back
Why did the learned physics model handle 9 balls but not stiffer springs? Why is an agent's "done" message not evidence?

In [ ]:
progress_report()